# Chapter 3: Unsupervised Learning and Preprocessing

**Sumber:** Introduction to Machine Learning with Python – Andreas C. Müller & Sarah Guido

## Tujuan Praktikum
Pada chapter ini, kita akan mempelajari:
- **Preprocessing dan Scaling** data (StandardScaler, MinMaxScaler, RobustScaler, Normalizer)
- **Dimensionality Reduction**: PCA, NMF, t-SNE
- **Clustering**: k-Means, Agglomerative Clustering, DBSCAN
- Cara evaluasi dan perbandingan metode clustering


## 1. Persiapan Environment

In [ ]:
# Install library yang diperlukan
!pip install mglearn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import mglearn
from sklearn.model_selection import train_test_split

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 2. Types of Unsupervised Learning

Dua jenis utama unsupervised learning:
1. **Transformations of the dataset**: membuat representasi baru dari data
   - Dimensionality reduction (PCA, NMF, t-SNE)
2. **Clustering**: mempartisi data ke dalam kelompok-kelompok yang serupa
   - k-Means, Agglomerative, DBSCAN


## 3. Challenges in Unsupervised Learning

Tantangan utama dalam unsupervised learning adalah **evaluasi** — kita tidak memiliki label yang benar.
Sulit untuk mengetahui apakah model belajar sesuatu yang berguna.

Unsupervised learning sering digunakan sebagai:
- Preprocessing sebelum supervised learning
- Eksplorasi data (EDA)


## 4. Preprocessing and Scaling

Kebanyakan algoritma ML sensitif terhadap skala fitur. Preprocessing membantu memastikan
semua fitur memiliki pengaruh yang setara.


In [ ]:
mglearn.plots.plot_scaling()
plt.title("Figure 3-1: Scaling of datasets")
plt.show()

### 4.1 Different Kinds of Preprocessing

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, Normalizer

# Data contoh untuk visualisasi
X_train = np.array([[ 0,  1,  2,  3],
                    [ 4,  5,  6,  7],
                    [ 8,  9, 10, 11]])

# MinMaxScaler
scaler = MinMaxScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
print("=== MinMaxScaler ===")
print("Scaled training data:\n{}".format(X_train_scaled))
print("per-feature minimum: {}".format(X_train_scaled.min(axis=0)))
print("per-feature maximum: {}".format(X_train_scaled.max(axis=0)))

In [ ]:
# StandardScaler
scaler = StandardScaler()
scaler.fit(X_train)
X_train_std = scaler.transform(X_train)
print("=== StandardScaler ===")
print("Scaled training data:\n{}".format(X_train_std))
print("per-feature mean (after scaling): {}".format(X_train_std.mean(axis=0)))
print("per-feature std (after scaling): {}".format(X_train_std.std(axis=0)))

### 4.2 Comparing Scalers

In [ ]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()

fig, axes = plt.subplots(15, 2, figsize=(10, 20))

ax = axes.ravel()

for i in range(30):
    _, bins = np.histogram(cancer.data[:, i], bins=50)
    ax[i].hist(cancer.data[cancer.target == 0][:, i], bins=bins,
               color=mglearn.cm3(0), alpha=.5)
    ax[i].hist(cancer.data[cancer.target == 1][:, i], bins=bins,
               color=mglearn.cm3(2), alpha=.5)
    ax[i].set_title(cancer.feature_names[i])
    ax[i].set_yticks(())

ax[0].set_xlabel("Feature magnitude")
ax[0].set_ylabel("Frequency")
ax[0].legend(["malignant", "benign"], loc="best")
fig.tight_layout()
plt.suptitle("Figure 3-2: Histograms of cancer dataset features", y=1.01)
plt.show()

### 4.3 Applying Data Transformations

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

cancer = load_breast_cancer()

X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, random_state=1)

print("X_train shape: {}".format(X_train.shape))
print("X_test shape: {}".format(X_test.shape))

In [ ]:
scaler = MinMaxScaler()
scaler.fit(X_train)
print("Scaler object: {}".format(scaler))

In [ ]:
# transform data
X_train_scaled = scaler.transform(X_train)
# print dataset properties before and after scaling
print("transformed shape: {}".format(X_train_scaled.shape))
print("per-feature minimum before scaling:\n {}".format(X_train.min(axis=0)))
print("per-feature maximum before scaling:\n {}".format(X_train.max(axis=0)))

In [ ]:
print("per-feature minimum after scaling:\n {}".format(
    X_train_scaled.min(axis=0)))
print("per-feature maximum after scaling:\n {}".format(
    X_train_scaled.max(axis=0)))

In [ ]:
# transform test data
X_test_scaled = scaler.transform(X_test)
# print test data properties after scaling
print("per-feature minimum after scaling:\n{}".format(X_test_scaled.min(axis=0)))
print("per-feature maximum after scaling:\n{}".format(X_test_scaled.max(axis=0)))

### 4.4 Scaling Training and Test Data the Same Way

**PENTING**: Selalu fit scaler pada data training saja, kemudian transform baik training maupun test data.
Jangan fit scaler pada test data! Ini akan menyebabkan **data leakage**.


In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Contoh yang BENAR
scaler = MinMaxScaler()
scaler.fit(X_train)                   # Fit HANYA pada training data
X_train_scaled = scaler.transform(X_train)   # Transform training data
X_test_scaled = scaler.transform(X_test)     # Transform test data menggunakan scaler yang sama

print("Training data min:", X_train_scaled.min(axis=0)[:5])
print("Test data min:", X_test_scaled.min(axis=0)[:5])  # Mungkin < 0 atau > 1, itu normal!

### 4.5 The Effect of Preprocessing on Supervised Learning

In [ ]:
from sklearn.svm import SVC
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

cancer = load_breast_cancer()

X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, random_state=0)

# SVM tanpa scaling
svm = SVC(C=100)
svm.fit(X_train, y_train)
print("Test set accuracy: {:.2f}".format(svm.score(X_test, y_test)))

In [ ]:
# SVM dengan MinMaxScaler
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# learn an SVM on the scaled training data
svm.fit(X_train_scaled, y_train)

# score on the scaled test set
print("Scaled test set accuracy: {:.2f}".format(
    svm.score(X_test_scaled, y_test)))

In [ ]:
# SVM dengan StandardScaler
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm.fit(X_train_scaled, y_train)
print("SVM test accuracy (StandardScaler): {:.2f}".format(
    svm.score(X_test_scaled, y_test)))

## 5. Dimensionality Reduction, Feature Extraction, and Manifold Learning

### 5.1 Principal Component Analysis (PCA)

PCA adalah metode dimensionality reduction yang paling umum.
PCA menemukan arah (principal components) yang menangkap varians terbesar dalam data,
kemudian memproyeksikan data ke subspace berdimensi lebih rendah.


In [ ]:
mglearn.plots.plot_pca_illustration()
plt.suptitle("Figure 3-3: PCA rotation of the input features", y=1.01)
plt.show()

In [ ]:
# PCA pada cancer dataset - visualisasi 2D
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()

# Before PCA - visualization on first two features only
fig, axes = plt.subplots(15, 2, figsize=(10, 20))

malignant = cancer.data[cancer.target == 0]
benign = cancer.data[cancer.target == 1]

ax = axes.ravel()

for i in range(30):
    _, bins = np.histogram(cancer.data[:, i], bins=50)
    ax[i].hist(malignant[:, i], bins=bins, color=mglearn.cm3(0), alpha=.5)
    ax[i].hist(benign[:, i], bins=bins, color=mglearn.cm3(2), alpha=.5)
    ax[i].set_title(cancer.feature_names[i])
    ax[i].set_yticks(())
ax[0].set_xlabel("Feature magnitude")
ax[0].set_ylabel("Frequency")
ax[0].legend(["malignant", "benign"], loc="best")
fig.tight_layout()
plt.show()

In [ ]:
# Standarisasi data sebelum PCA
scaler = StandardScaler()
scaler.fit(cancer.data)
X_scaled = scaler.transform(cancer.data)

# Simpan 2 principal component pertama
pca = PCA(n_components=2)
pca.fit(X_scaled)

# Transform data ke 2 komponen pertama
X_pca = pca.transform(X_scaled)

print("Original shape: {}".format(str(X_scaled.shape)))
print("Reduced shape: {}".format(str(X_pca.shape)))

In [ ]:
# Plot dari 2 principal component pertama
plt.figure(figsize=(8, 8))
mglearn.discrete_scatter(X_pca[:, 0], X_pca[:, 1], cancer.target)
plt.legend(cancer.target_names, loc="best")
plt.gca().set_aspect("equal")
plt.xlabel("First principal component")
plt.ylabel("Second principal component")
plt.title("Figure 3-4: First two principal components of the cancer dataset")
plt.show()

In [ ]:
print("PCA component shape: {}".format(pca.components_.shape))

In [ ]:
print("PCA components:\n{}".format(pca.components_))

In [ ]:
# Visualisasi heatmap komponen PCA
plt.matshow(pca.components_, cmap='viridis')
plt.yticks([0, 1], ["First component", "Second component"])
plt.colorbar()
plt.xticks(range(len(cancer.feature_names)), cancer.feature_names, rotation=60, ha='left')
plt.xlabel("Feature")
plt.ylabel("Principal components")
plt.title("Figure 3-5: Heatmap of PCA components on cancer dataset")
plt.tight_layout()
plt.show()

#### PCA pada dataset Faces (Eigenfaces)

In [ ]:
from sklearn.datasets import fetch_lfw_people

# Download/load dataset faces
try:
    people = fetch_lfw_people(min_faces_per_person=20, resize=0.7)
    image_shape = people.images[0].shape

    fig, axes = plt.subplots(2, 5, figsize=(15, 8),
                             subplot_kw={'xticks': (), 'yticks': ()})
    for target, image, ax in zip(people.target, people.images, axes.ravel()):
        ax.imshow(image)
        ax.set_title(people.target_names[target])
    plt.suptitle("Figure 3-6: Sample images from the LFW dataset")
    plt.tight_layout()
    plt.show()
    print("data shape: {}".format(people.images.shape))
    print("number of images per target: {}".format(np.bincount(people.target)))
    
except Exception as e:
    print(f"Dataset tidak dapat didownload: {e}")
    print("Menggunakan Olivetti Faces sebagai alternatif...")
    from sklearn.datasets import fetch_olivetti_faces
    faces_data = fetch_olivetti_faces()
    people = faces_data
    image_shape = people.images[0].shape
    print("Olivetti Faces shape:", people.images.shape)
    
    fig, axes = plt.subplots(2, 5, figsize=(15, 8),
                             subplot_kw={'xticks': (), 'yticks': ()})
    for target, image, ax in zip(people.target, people.images[:10], axes.ravel()):
        ax.imshow(image, cmap='gray')
        ax.set_title("Person {}".format(target))
    plt.suptitle("Sample images from Olivetti Faces dataset")
    plt.tight_layout()
    plt.show()

In [ ]:
# Batasi tiap orang maksimal 50 gambar (seperti buku)
try:
    mask = np.zeros(people.target.shape, dtype=bool)
    for target in np.unique(people.target):
        mask[np.where(people.target == target)[0][:50]] = 1

    X_people = people.data[mask]
    y_people = people.target[mask]

    # scale the grayscale values to be between 0 and 1
    # instead of 0 and 255 for better numeric stability
    X_people = X_people / 255.

    print("X_people shape:", X_people.shape)
    print("y_people shape:", y_people.shape)
except:
    # For olivetti faces (already normalized)
    X_people = people.data
    y_people = people.target
    print("X_people shape:", X_people.shape)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_people, y_people, stratify=y_people, random_state=0)

# Baseline: 1-NN menggunakan raw pixels
knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(X_train, y_train)
print("Test set score of 1-nn: {:.2f}".format(knn.score(X_test, y_test)))

In [ ]:
# PCA Whitening
from sklearn.decomposition import PCA

pca = PCA(n_components=100, whiten=True, random_state=0).fit(X_train)
X_train_pca = pca.transform(X_train)
X_test_pca = pca.transform(X_test)

print("X_train_pca.shape: {}".format(X_train_pca.shape))

In [ ]:
knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(X_train_pca, y_train)
print("Test set accuracy (with PCA): {:.2f}".format(knn.score(X_test_pca, y_test)))

In [ ]:
# Visualisasi eigenfaces
print("pca.components_.shape: {}".format(pca.components_.shape))

image_shape_local = people.images[0].shape
fig, axes = plt.subplots(3, 5, figsize=(15, 12),
                         subplot_kw={'xticks': (), 'yticks': ()})
for i, (component, ax) in enumerate(zip(pca.components_, axes.ravel())):
    ax.imshow(component.reshape(image_shape_local),
             cmap='viridis')
    ax.set_title("{}. component".format((i + 1)))
plt.suptitle("Figure 3-7: First 15 principal components (eigenfaces)")
plt.tight_layout()
plt.show()

In [ ]:
# Rekonstruksi wajah dari komponen PCA
mglearn.plots.plot_pca_faces(X_train, X_test, image_shape_local)
plt.suptitle("Figure 3-8: Reconstructing face images using increasing number of PCA components")
plt.show()

In [ ]:
# Scatter plot menggunakan 2 komponen PCA pertama
mglearn.discrete_scatter(X_train_pca[:, 0], X_train_pca[:, 1], y_train)
plt.xlabel("First principal component")
plt.ylabel("Second principal component")
plt.title("Figure 3-9: Scatter plot using first two principal components of faces")
plt.show()

### 5.2 Non-Negative Matrix Factorization (NMF)

NMF adalah metode lain untuk dimensionality reduction. Tidak seperti PCA:
- Semua komponen dan koefisien **non-negatif**
- Berguna untuk data yang naturally non-negatif (gambar, audio, teks)
- Komponen lebih mudah diinterpretasi


In [ ]:
mglearn.plots.plot_nmf_illustration()
plt.suptitle("Figure 3-10: Components found by NMF with 1 and 2 components")
plt.show()

In [ ]:
from sklearn.decomposition import NMF

nmf = NMF(n_components=15, random_state=0)
nmf.fit(X_train)
X_train_nmf = nmf.transform(X_train)
X_test_nmf = nmf.transform(X_test)

fig, axes = plt.subplots(3, 5, figsize=(15, 12),
                         subplot_kw={'xticks': (), 'yticks': ()})
for i, (component, ax) in enumerate(zip(nmf.components_, axes.ravel())):
    ax.imshow(component.reshape(image_shape_local))
    ax.set_title("{}. component".format(i))
plt.suptitle("Figure 3-12: Components found by NMF with 15 components on faces dataset")
plt.tight_layout()
plt.show()

In [ ]:
# Contoh NMF pada data sinusoidal
S = mglearn.datasets.make_signals()
plt.figure(figsize=(6, 1))
plt.plot(S, '-')
plt.xlabel("Time")
plt.ylabel("Signal")
plt.margins(0)
plt.title("Figure 3-14: Original signals")
plt.show()

In [ ]:
# Mix data
A = np.random.RandomState(0).uniform(size=(100, 3))
X = np.dot(S, A.T)
print("Shape of measurements: {}".format(X.shape))

nmf = NMF(n_components=3, random_state=42)
S_ = nmf.fit_transform(X)
print("Recovered signal shape: {}".format(S_.shape))

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=3)
H = pca.fit_transform(X)

models = [X, S, S_, H]
names = ['Observations (first three measurements)',
         'True sources',
         'NMF recovered signals',
         'PCA recovered signals']

fig, axes = plt.subplots(4, figsize=(8, 4), gridspec_kw={'hspace': .5},
                         subplot_kw={'xticks': (), 'yticks': ()})

for model, name, ax in zip(models, names, axes):
    ax.set_title(name)
    ax.plot(model[:, :3], '-')
    ax.margins(0)
plt.suptitle("Figure 3-15: Signals recovered by NMF and PCA")
plt.show()

### 5.3 Manifold Learning with t-SNE

**t-SNE (t-distributed Stochastic Neighbor Embedding)** adalah metode manifold learning
yang sangat baik untuk visualisasi data berdimensi tinggi dalam 2D.

t-SNE mempertahankan struktur lokal data lebih baik dari PCA.
**Catatan**: t-SNE hanya untuk visualisasi, tidak untuk preprocessing supervised learning.


In [ ]:
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE

digits = load_digits()

fig, axes = plt.subplots(2, 5, figsize=(10, 5),
                         subplot_kw={'xticks': (), 'yticks': ()})
for ax, img in zip(axes.ravel(), digits.images):
    ax.imshow(img)
plt.suptitle("Figure 3-16: Sample images from the digits dataset")
plt.tight_layout()
plt.show()

In [ ]:
# Visualisasi dengan PCA
pca = PCA(n_components=2)
pca.fit(digits.data)
digits_pca = pca.transform(digits.data)

colors = ["#476A2A", "#7851B8", "#BD3430", "#4A2D4E", "#875418",
          "#A83683", "#4E655E", "#853541", "#3A3120","#535D8E"]

plt.figure(figsize=(10, 10))
plt.xlim(digits_pca[:, 0].min(), digits_pca[:, 0].max())
plt.ylim(digits_pca[:, 1].min(), digits_pca[:, 1].max())

for i in range(len(digits.data)):
    plt.text(digits_pca[i, 0], digits_pca[i, 1], str(digits.target[i]),
             color=colors[digits.target[i]],
             fontdict={'weight': 'bold', 'size': 9})

plt.xlabel("First principal component")
plt.ylabel("Second principal component")
plt.title("Figure 3-17: PCA visualization of the digits dataset")
plt.show()

In [ ]:
# Visualisasi dengan t-SNE
tsne = TSNE(random_state=42)
# use fit_transform instead of fit, as TSNE has no transform method
digits_tsne = tsne.fit_transform(digits.data)

In [ ]:
plt.figure(figsize=(10, 10))
plt.xlim(digits_tsne[:, 0].min(), digits_tsne[:, 0].max() + 1)
plt.ylim(digits_tsne[:, 1].min(), digits_tsne[:, 1].max() + 1)

for i in range(len(digits.data)):
    plt.text(digits_tsne[i, 0], digits_tsne[i, 1], str(digits.target[i]),
             color=colors[digits.target[i]],
             fontdict={'weight': 'bold', 'size': 9})

plt.xlabel("t-SNE feature 0")
plt.ylabel("t-SNE feature 1")
plt.title("Figure 3-18: t-SNE visualization of the digits dataset")
plt.show()

## 6. Clustering

Clustering adalah mempartisi dataset ke dalam kelompok-kelompok (clusters) yang memiliki
sampel serupa. Tidak seperti classification, kita tidak menentukan kelas di awal —
algoritma yang menemukan sendiri struktur pengelompokan dari data.


### 6.1 k-Means Clustering

k-Means adalah algoritma clustering yang paling populer.
Menemukan k cluster centers yang merepresentasikan pusat dari cluster.

**Algoritma**:
1. Inisialisasi k cluster centers secara acak
2. Assign tiap data point ke cluster center terdekat
3. Update cluster center sebagai rata-rata data point di cluster tersebut
4. Ulangi 2-3 sampai konvergen


In [ ]:
mglearn.plots.plot_kmeans_algorithm()
plt.title("Figure 3-19: Input data and three steps of k-means algorithm")
plt.show()

In [ ]:
mglearn.plots.plot_kmeans_boundaries()
plt.title("Figure 3-20: Cluster assignments and cluster centers found by k-means")
plt.show()

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

# generate some data
X, y = make_blobs(random_state=1)

# build the clustering model
kmeans = KMeans(n_clusters=3)
kmeans.fit(X)

print("Cluster memberships:\n{}".format(kmeans.labels_))

In [ ]:
print("Predictions:\n{}".format(kmeans.predict(X)))

In [ ]:
mglearn.discrete_scatter(X[:, 0], X[:, 1], kmeans.labels_, markers='o')
mglearn.discrete_scatter(kmeans.cluster_centers_[:, 0],
                         kmeans.cluster_centers_[:, 1], [0, 1, 2],
                         markers='^', markeredgewidth=2)
plt.xlabel("Feature 0")
plt.ylabel("Feature 1")
plt.title("Figure 3-21: Cluster assignments and centers found by k-means (3 clusters)")
plt.show()

In [ ]:
# k-Means dengan berbagai jumlah cluster
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# using two cluster centers:
kmeans = KMeans(n_clusters=2)
kmeans.fit(X)
assignments = kmeans.labels_

mglearn.discrete_scatter(X[:, 0], X[:, 1], assignments, ax=axes[0])
axes[0].set_title("2 Clusters")

# using five cluster centers:
kmeans = KMeans(n_clusters=5)
kmeans.fit(X)
assignments = kmeans.labels_

mglearn.discrete_scatter(X[:, 0], X[:, 1], assignments, ax=axes[1])
axes[1].set_title("5 Clusters")

plt.suptitle("Figure 3-22: Cluster assignments for 2 and 5 clusters")
plt.tight_layout()
plt.show()

#### Kelemahan k-Means

In [ ]:
# k-Means tidak bekerja baik untuk semua bentuk data
X_varied, y_varied = make_blobs(n_samples=200,
                                 cluster_std=[1.0, 2.5, 0.5],
                                 random_state=170)
y_pred = KMeans(n_clusters=3, random_state=0).fit_predict(X_varied)

mglearn.discrete_scatter(X_varied[:, 0], X_varied[:, 1], y_pred)
plt.legend(["cluster 0", "cluster 1", "cluster 2"], loc='best')
plt.xlabel("Feature 0")
plt.ylabel("Feature 1")
plt.title("Figure 3-23: k-means clustering with different cluster densities")
plt.show()

In [ ]:
# Anisotropically distributed data - k-Means gagal
X, y = make_blobs(random_state=170, n_samples=600)
rng = np.random.RandomState(74)

# transform the data to be stretched
transformation = rng.normal(size=(2, 2))
X = np.dot(X, transformation)

# cluster the data into three clusters
kmeans = KMeans(n_clusters=3)
kmeans.fit(X)
y_pred = kmeans.predict(X)

# plot the cluster assignments and cluster centers
plt.scatter(X[:, 0], X[:, 1], c=y_pred, cmap=mglearn.cm3)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            marker='^', c=[0, 1, 2], s=100, linewidth=2, cmap=mglearn.cm3)
plt.xlabel("Feature 0")
plt.ylabel("Feature 1")
plt.title("Figure 3-24: k-means fails on anisotropically distributed data")
plt.show()

In [ ]:
# Two-moons dataset - k-Means gagal
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=200, noise=0.05, random_state=0)

# Cluster the data into two clusters
kmeans = KMeans(n_clusters=2)
kmeans.fit(X)
y_pred = kmeans.predict(X)

# Plot the cluster assignments and cluster centers
plt.scatter(X[:, 0], X[:, 1], c=y_pred, cmap=mglearn.cm2, s=60)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            marker='^', c=[mglearn.cm2(0), mglearn.cm2(1)], s=100, linewidth=2)
plt.xlabel("Feature 0")
plt.ylabel("Feature 1")
plt.title("Figure 3-25: k-means fails on two-moons dataset")
plt.show()

#### k-Means untuk Vector Quantization

In [ ]:
# k-Means untuk kompresi gambar (vector quantization)
from sklearn.datasets import load_sample_image
from sklearn.cluster import KMeans
import numpy as np

try:
    china = load_sample_image("china.jpg")
    ax = plt.axes(xticks=[], yticks=[])
    ax.imshow(china)
    plt.title("Original image (china)")
    plt.show()

    print("Image shape: {}".format(china.shape))

    # Convert to float, normalize
    X_img = china.reshape(-1, 3)
    print("Number of pixels:", X_img.shape[0])
    
    # KMeans clustering
    from sklearn.cluster import MiniBatchKMeans
    
    n_colors = 64
    kmeans_img = MiniBatchKMeans(n_clusters=n_colors, random_state=0)
    kmeans_img.fit(X_img)
    
    # Reconstruct image
    china_recolored = kmeans_img.cluster_centers_[kmeans_img.labels_]
    china_recolored = china_recolored.reshape(china.shape).astype(np.uint8)
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(china)
    axes[0].set_title("Original image")
    axes[1].imshow(china_recolored)
    axes[1].set_title("Quantized image ({} colors)".format(n_colors))
    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])
    plt.suptitle("Figure 3-26: Vector quantization with k-means")
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print("Sample image tidak tersedia:", e)
    print("Membuat visualisasi dengan synthetic data...")
    X, y = make_blobs(random_state=42, n_samples=200)
    kmeans_vq = KMeans(n_clusters=5, random_state=0)
    kmeans_vq.fit(X)
    mglearn.discrete_scatter(X[:, 0], X[:, 1], kmeans_vq.labels_)
    plt.title("k-Means Vector Quantization (synthetic data)")
    plt.show()

### 6.2 Agglomerative Clustering

Agglomerative Clustering adalah hierarki clustering yang bekerja secara **bottom-up**:
1. Mulai: tiap data point adalah cluster tersendiri
2. Secara iteratif, gabungkan dua cluster yang paling mirip
3. Hentikan ketika jumlah cluster yang diinginkan tercapai

Parameter utama: **linkage** (cara mengukur jarak antar cluster):
- `ward` (default): meminimalkan variance dalam cluster
- `average`: rata-rata jarak antar titik
- `complete` (maximum linkage): jarak maksimum antar titik


In [ ]:
mglearn.plots.plot_agglomerative_algorithm()
plt.title("Figure 3-27: The agglomerative clustering algorithm on a simple dataset")
plt.show()

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import make_blobs

X, y = make_blobs(random_state=1)

agg = AgglomerativeClustering(n_clusters=3)
assignment = agg.fit_predict(X)

mglearn.discrete_scatter(X[:, 0], X[:, 1], assignment)
plt.legend(["Cluster 0", "Cluster 1", "Cluster 2"], loc="best")
plt.xlabel("Feature 0")
plt.ylabel("Feature 1")
plt.title("Figure 3-28: Cluster assignments from agglomerative clustering with 3 clusters")
plt.show()

#### Dendrogram - Hierarki Clustering

In [ ]:
# Dendrogram menggunakan scipy
from scipy.cluster.hierarchy import dendrogram, ward

X, y = make_blobs(random_state=0, n_samples=12)
# Apply the ward clustering to the data array X
# The SciPy ward function returns an array that specifies which
# cluster each data point belongs to
linkage_array = ward(X)
# Now we plot the dendrogram for the linkage_array containing the distances
# between clusters
dendrogram(linkage_array)

# Mark the cuts in the tree that signify two or three clusters
ax = plt.gca()
bounds = ax.get_xbound()
ax.plot(bounds, [7.25, 7.25], '--', c='k')
ax.plot(bounds, [4, 4], '--', c='k')
ax.text(bounds[1], 7.25, ' two clusters', va='center', fontdict={'size': 15})
ax.text(bounds[1], 4, ' three clusters', va='center', fontdict={'size': 15})
plt.xlabel("Sample index")
plt.ylabel("Cluster distance")
plt.title("Figure 3-29: Dendrogram of agglomerative clustering")
plt.show()

#### Perbandingan Linkage untuk Agglomerative Clustering

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import make_blobs, make_moons

X, y = make_blobs(random_state=0, n_samples=12)
for linkage in ('ward', 'average', 'complete', 'single'):
    clustering = AgglomerativeClustering(n_clusters=3, linkage=linkage)
    clustering.fit(X)
    print("Linkage {}: {}".format(linkage, clustering.labels_))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3))

for ax, linkage in zip(axes, ('ward', 'average', 'complete', 'single')):
    X_moons, y_moons = make_moons(n_samples=200, noise=0.05, random_state=0)
    agg = AgglomerativeClustering(n_clusters=2, linkage=linkage)
    agg.fit(X_moons)
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=agg.labels_, s=30)
    ax.set_title("Linkage: {}".format(linkage))
    ax.set_xticks([])
    ax.set_yticks([])
plt.suptitle("Figure 3-32: Comparison of linkage options on two-moons dataset")
plt.tight_layout()
plt.show()

### 6.3 DBSCAN (Density-Based Spatial Clustering of Applications with Noise)

DBSCAN adalah algoritma clustering berbasis density.
- Tidak perlu menentukan jumlah cluster di awal
- Dapat menemukan cluster berbentuk arbitrary
- Dapat mengidentifikasi **noise points** (outlier)

Parameter kunci:
- **eps**: jarak maksimum antar titik untuk dianggap tetangga
- **min_samples**: jumlah minimum sample dalam lingkungan eps untuk dianggap core point

Tipe titik dalam DBSCAN:
- **Core point**: memiliki >= min_samples tetangga dalam radius eps
- **Border point**: dalam radius eps core point tapi memiliki < min_samples tetangga sendiri
- **Noise point**: bukan core point maupun border point


In [ ]:
mglearn.plots.plot_dbscan()
plt.title("Figure 3-33: Cluster assignments by DBSCAN with different eps and min_samples")
plt.show()

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_blobs, make_moons

X, y = make_blobs(random_state=0, n_samples=12)

dbscan = DBSCAN()
clusters = dbscan.fit_predict(X)
print("Cluster memberships:\n{}".format(clusters))

In [ ]:
# DBSCAN pada two-moons dataset
X, y = make_moons(n_samples=200, noise=0.05, random_state=0)

# Rescale the data to zero mean and unit variance
scaler = StandardScaler()
scaler.fit(X)
X_scaled = scaler.transform(X)

dbscan = DBSCAN()
clusters = dbscan.fit_predict(X_scaled)
# plot the cluster assignments
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=clusters, cmap=mglearn.cm2, s=60)
plt.xlabel("Feature 0")
plt.ylabel("Feature 1")
plt.title("Figure 3-35: DBSCAN clusters on scaled two-moons dataset")
plt.show()

In [ ]:
# Efek parameter eps dan min_samples pada DBSCAN
fig, axes = plt.subplots(3, 4, figsize=(11, 8),
                          subplot_kw={'xticks': (), 'yticks': ()})

# create a list of parameter settings to try
params = {'eps': [1, 1.5, 2, 3], 'min_samples': [2, 3, 5]}

for ax_row, min_samples in zip(axes, params['min_samples']):
    for ax, eps in zip(ax_row, params['eps']):
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        clusters = dbscan.fit_predict(X_scaled)
        ax.scatter(X_scaled[:, 0], X_scaled[:, 1], c=clusters,
                   cmap=mglearn.cm2_cycle, s=60)
        ax.set_title("eps={:.1f} min_samples={:d}".format(eps, min_samples))

plt.suptitle("Figure 3-36: DBSCAN clusters for different eps and min_samples")
plt.tight_layout()
plt.show()

## 7. Comparing and Evaluating Clustering Algorithms

### 7.1 Evaluating Clustering with Ground Truth

Ketika kita memiliki label ground truth, kita bisa menggunakan:
- **Adjusted Rand Index (ARI)**: mengukur kesamaan clustering, rentang -1 hingga 1
- **Normalized Mutual Information (NMI)**: mengukur informasi bersama antara dua clustering


In [ ]:
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler

X, y = make_moons(n_samples=200, noise=0.05, random_state=0)
scaler = StandardScaler()
scaler.fit(X)
X_scaled = scaler.transform(X)

fig, axes = plt.subplots(1, 4, figsize=(15, 3),
                         subplot_kw={'xticks': (), 'yticks': ()})

# make a list of algorithms to use
algorithms = [KMeans(n_clusters=2),
              AgglomerativeClustering(n_clusters=2),
              DBSCAN()]

# create a random cluster assignment for reference
random_state = np.random.RandomState(seed=0)
random_clusters = random_state.randint(low=0, high=2, size=len(X))

# plot random assignment
axes[0].scatter(X_scaled[:, 0], X_scaled[:, 1], c=random_clusters,
                cmap=mglearn.cm3, s=60)
axes[0].set_title("Random assignment - ARI: {:.2f}".format(
    adjusted_rand_score(y, random_clusters)))

for ax, algorithm in zip(axes[1:], algorithms):
    # plot the cluster assignments and cluster centers
    clusters = algorithm.fit_predict(X_scaled)
    ax.scatter(X_scaled[:, 0], X_scaled[:, 1], c=clusters,
               cmap=mglearn.cm3, s=60)
    ax.set_title("{} - ARI: {:.2f}".format(algorithm.__class__.__name__,
                                            adjusted_rand_score(y, clusters)))

plt.suptitle("Figure 3-37: Comparing clustering algorithms using Adjusted Rand Index")
plt.tight_layout()
plt.show()

### 7.2 Evaluating Clustering Without Ground Truth

In [ ]:
from sklearn.metrics.cluster import silhouette_score
from sklearn.datasets import make_blobs

X, y = make_blobs(random_state=0, n_samples=100)

# compare two clustering algorithms using silhouette score
scores = []
for k in range(2, 8):
    kmeans = KMeans(n_clusters=k, random_state=0)
    labels = kmeans.fit_predict(X)
    score = silhouette_score(X, labels)
    scores.append(score)
    print("k={} - Silhouette score: {:.3f}".format(k, score))

plt.plot(range(2, 8), scores, 'o-')
plt.xlabel("Number of clusters")
plt.ylabel("Silhouette score")
plt.title("Silhouette scores for different number of clusters")
plt.show()

### 7.3 Comparing Clustering Algorithms on Various Datasets

In [ ]:
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.preprocessing import StandardScaler
import matplotlib as mpl

# Generate datasets
n_samples = 1500
datasets = [
    make_blobs(n_samples=n_samples, random_state=170),
    make_circles(n_samples=n_samples, factor=.5, noise=.05),
    make_moons(n_samples=n_samples, noise=.05)
]
dataset_names = ["Blobs", "Circles", "Moons"]

clustering_algorithms = [
    ('KMeans', KMeans(n_clusters=2, random_state=0)),
    ('Agglomerative (ward)', AgglomerativeClustering(n_clusters=2)),
    ('DBSCAN', DBSCAN(eps=0.3, min_samples=5))
]

fig, axes = plt.subplots(len(datasets), len(clustering_algorithms),
                          figsize=(12, 9))

for i, (X_data, y_data) in enumerate(datasets):
    # standardize
    X_scaled = StandardScaler().fit_transform(X_data)
    for j, (name, algo) in enumerate(clustering_algorithms):
        labels = algo.fit_predict(X_scaled)
        ax = axes[i][j]
        ax.scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels,
                   cmap='viridis', s=10)
        if i == 0:
            ax.set_title(name, fontsize=12)
        if j == 0:
            ax.set_ylabel(dataset_names[i], fontsize=12)
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle("Figure 3-38: Comparison of clustering algorithms on various datasets")
plt.tight_layout()
plt.show()

### 7.4 Clustering on the Digits Dataset

In [ ]:
from sklearn.datasets import load_digits
from sklearn.cluster import KMeans
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score

digits = load_digits()

# Cluster dengan k=10 (karena ada 10 digit)
km = KMeans(n_clusters=10, random_state=0)
clusters_digits = km.fit_predict(digits.data)

print("Adjusted Rand index: {:.2f}".format(
    adjusted_rand_score(digits.target, clusters_digits)))
print("NMI: {:.2f}".format(
    normalized_mutual_info_score(digits.target, clusters_digits)))

In [ ]:
# Tabel silang: clustering vs label sebenarnya
from sklearn.metrics.cluster import contingency_matrix

contingency = contingency_matrix(digits.target, clusters_digits)
print("Contingency matrix:")
print(contingency)

In [ ]:
# Visualisasi cluster centers
fig, axes = plt.subplots(2, 5, subplot_kw={'xticks': (), 'yticks': ()},
                         figsize=(12, 4))
for center, ax in zip(km.cluster_centers_, axes.ravel()):
    ax.imshow(center.reshape(8, 8), cmap='gray_r')
    
plt.suptitle("Figure 3-39: Cluster centers found by k-means on the digits dataset")
plt.tight_layout()
plt.show()

In [ ]:
# Clustering dengan preprocessing PCA
from sklearn.decomposition import PCA

# Reducere dimensi menggunakan PCA
pca = PCA(n_components=2)
digits_pca = pca.fit_transform(digits.data)

colors = ["#476A2A", "#7851B8", "#BD3430", "#4A2D4E", "#875418",
          "#A83683", "#4E655E", "#853541", "#3A3120", "#535D8E"]

plt.figure(figsize=(10, 10))
plt.xlim(digits_pca[:, 0].min(), digits_pca[:, 0].max() + 1)
plt.ylim(digits_pca[:, 1].min(), digits_pca[:, 1].max() + 1)
for i in range(len(digits.data)):
    plt.text(digits_pca[i, 0], digits_pca[i, 1], str(digits.target[i]),
             color=colors[digits.target[i]],
             fontdict={'weight': 'bold', 'size': 9})
plt.xlabel("First principal component")
plt.ylabel("Second principal component")
plt.title("Figure 3-40: PCA visualization of digits clustering")
plt.show()

In [ ]:
# k-Means clustering pada data setelah PCA preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score

# Standarisasi
scaler = StandardScaler()
X_scaled = scaler.fit_transform(digits.data)

# PCA
pca = PCA(n_components=10)
X_pca = pca.fit_transform(X_scaled)

# k-Means pada PCA space
km_pca = KMeans(n_clusters=10, random_state=0)
labels_pca = km_pca.fit_predict(X_pca)

print("ARI with PCA preprocessing: {:.2f}".format(
    adjusted_rand_score(digits.target, labels_pca)))
print("NMI with PCA preprocessing: {:.2f}".format(
    normalized_mutual_info_score(digits.target, labels_pca)))

## 8. Summary of Clustering Methods

### Perbandingan Metode Clustering

| Metode | Kelebihan | Kekurangan | Parameter Utama |
|--------|-----------|------------|-----------------|
| **k-Means** | Sederhana, cepat, scalable | Asumsi cluster bundar, perlu tentukan k | n_clusters |
| **Agglomerative** | Tidak asumsi bentuk cluster, menghasilkan dendrogram | Lambat O(n²), tidak bisa predict data baru | n_clusters, linkage |
| **DBSCAN** | Tidak perlu k, deteksi noise, arbitrary shape | Sensitif eps & min_samples, buruk pada density berbeda | eps, min_samples |

### Ringkasan Chapter 3

1. **Preprocessing**: Gunakan `StandardScaler` atau `MinMaxScaler` - SELALU fit hanya pada training data
2. **PCA**: Untuk dimensionality reduction dan visualisasi - perlu scaling terlebih dahulu  
3. **NMF**: Alternatif PCA untuk data non-negatif - komponen lebih interpretable
4. **t-SNE**: Terbaik untuk visualisasi 2D - tidak untuk preprocessing
5. **k-Means**: Pilihan default yang baik - cepat dan efisien
6. **Agglomerative**: Ketika ingin hierarki atau bentuk cluster non-bundar
7. **DBSCAN**: Ketika ada noise/outlier atau bentuk cluster tidak beraturan
